# 02 — LLM Annotation
Three open-weight models (Qwen3-14B, Gemma-3-12B-IT, Llama-3.1-8B-Instruct, 4-bit NF4) annotate every pair three times (temperature 0.7, top-p 0.9, fixed per-run seeds).

* **Stage 1** receives the context and previous utterance (not the response) and returns the Expected Response Function.
* **Stage 2** receives the context, previous utterance, response and the model's own Stage-1 label, and returns Actual Function, Fulfillment and Mismatch Type.

The notebook resumes from `llm_annotations_master.csv`: runs that are already valid are reused. A GPU is required.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# 1. Install required packages
!pip -q install -U transformers accelerate bitsandbytes sentencepiece pandas


# 2. Import libraries
import os
import re
import gc
import json
import random
import hashlib
import numpy as np
import pandas as pd
import torch

from collections import Counter
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)


# 3. Check GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Enable a GPU in "
        "Runtime > Change runtime type > T4 GPU, then rerun."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)
print("PyTorch version:", torch.__version__)


# 4. Load input dataset
INPUT_FILE = "dialogue_expectation_mismatch.csv"

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}. "
        "Upload the CSV to the Colab session first."
    )

df = pd.read_csv(
    INPUT_FILE,
    dtype=str,
    keep_default_na=False
)

required_columns = [
    "pair_id",
    "context",
    "previous_utterance",
    "response"
]

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing input columns: {sorted(missing_columns)}"
    )

df = df.copy()

for col in required_columns:
    df[col] = df[col].fillna("").astype(str)

# Normalize pair IDs.
df["pair_id"] = df["pair_id"].str.strip()

# Validate pair IDs.
if (df["pair_id"] == "").any():
    raise ValueError("pair_id cannot be empty.")

if df["pair_id"].duplicated().any():
    raise ValueError(
        "pair_id must be unique for safe result tracking."
    )
# Keep only the fields that the LLM annotation prompts need.
# Original dialogue-act labels and other metadata are excluded.
annotation_df = df[required_columns].copy()

print("Total rows:", len(annotation_df))
print("Unique pair IDs:", annotation_df["pair_id"].nunique())

## Settings

In [ ]:
MODELS = {
    "Qwen3-14B": "Qwen/Qwen3-14B",
    "Gemma3-12B": "google/gemma-3-12b-it",
    "Llama3.1-8B": "meta-llama/Llama-3.1-8B-Instruct"
}

N_RUNS = 3

# MANUAL ROW RANGE — inclusive
START_ROW = 0
END_ROW = len(annotation_df) - 1

if not (0 <= START_ROW <= END_ROW < len(annotation_df)):
    raise ValueError(
        f"Invalid range. Dataset has {len(annotation_df)} rows."
    )

batch_df = annotation_df.iloc[START_ROW:END_ROW + 1].copy()

print(f"Annotating rows {START_ROW} to {END_ROW}, inclusive ({len(batch_df)} rows)")

OUTPUT_FILE = "llm_annotations_master.csv"

## Prompts and label sets

In [ ]:
EXPECTED_LABELS = {
    "Answer",
    "Accept/Reject",
    "Perform/Provide",
    "Clarification",
    "Acknowledgment"
}

ACTUAL_LABELS = {
    "Answer",
    "Accept/Reject",
    "Perform/Provide",
    "Clarification",
    "Acknowledgment",
    "Information Request"
}

FULFILLMENT_LABELS = {
    "Fulfilled",
    "Not Fulfilled",
    "Unjudgeable"
}

MISMATCH_LABELS = {
    "None",
    "Alternative",
    "Topic Shift",
    "Indirect Response",
    "Non-Response"
}


# ============================================================
# STAGE 1: EXPECTED RESPONSE FUNCTION
# ============================================================

STAGE1_SYSTEM_PROMPT = """
You are annotating expected response functions in everyday English conversations.

TASK
Identify the single response function most directly called for by the
PREVIOUS UTTERANCE.

You will receive:
- Context: an optional preceding utterance.
- Previous Utterance: the utterance whose expected response function
  must be classified.

You will NOT receive the actual response. Determine the Expected Function
using only the information available in this stage.

DATA INDEPENDENCE
Use only the dialogue text supplied in this prompt and the guidelines below.
Do not consult external sources or rely on dataset annotations, dialogue-act
labels, reference answers, metadata, or previously published classifications.
Do not infer a label from the identity or presumed source of the dialogue.

CONTEXT RULE
Classify the Previous Utterance.
Use Context only when necessary to resolve a reference or understand the
meaning of the Previous Utterance.
Context must not determine the label by itself.
If the Previous Utterance does not support a particular function, do not
assign that function merely because it seems plausible from Context.

EXPECTED FUNCTION TAXONOMY

1. Answer
The Previous Utterance seeks information, a fact, an explanation, or
confirmation about a state of affairs.
This includes open-ended and yes/no questions.
Examples include asking about someone's identity, location, plans,
readiness, availability, or the reason for something.

2. Accept/Reject
The Previous Utterance is an invitation, offer, proposal, or request
that primarily calls for a decision, agreement, refusal, or commitment.
For example, "Would you like to come with us?" or "Can you come over
at 11?" when the speaker is asking the listener to commit to doing so.
Do not use this label for every question that can be answered with yes or no.

3. Perform/Provide
The Previous Utterance asks or directs the listener to carry out a concrete
action or provide an item, service, or resource.
The expected response is the action or provision itself, not merely
a statement of willingness to do it.
For example, a direct instruction to bring an item calls for Perform/Provide.

4. Clarification
The Previous Utterance explicitly asks the listener to clarify or explain
something that is ambiguous, unclear, or misunderstood.
The expected response is an explanation, correction, or clarification
that resolves the ambiguity or misunderstanding.
Do not use this merely because the utterance contains a question.
A question seeking new information is Answer, not Clarification.

5. Acknowledgment
The Previous Utterance primarily presents information or makes a social
or conversational contribution for which recognition, acknowledgment,
or a conventional social response is most directly expected.
Examples include greetings, thanks, and apologies when no more specific
response function is directly called for.
Do not use Acknowledgment as a catch-all for statements.

DECISION RULES

- Select exactly one label from the five permitted labels.
- Determine the Expected Function independently of any actual response.
- Use the most specific label supported by the Previous Utterance.
- A question seeking factual information or confirmation is Answer,
  even if the answer could be yes or no.
- A question asking the listener to accept an invitation, agree to a proposal,
  or commit to an action is Accept/Reject.
- Distinguish a question about a fact from a request for commitment:
  "Are you free at 11?" generally seeks information and is Answer;
  "Can you come over at 11?" generally asks the listener to commit and is
  Accept/Reject when used as a request to come.
- A directive to carry out an action or provide something is Perform/Provide.
- Do not classify every request as Accept/Reject.
- Do not classify a concrete directive as Answer merely because the listener
  could respond verbally.
- Use Clarification when the Previous Utterance asks the listener to resolve
  ambiguity or misunderstanding by explaining or correcting something.
- Do not classify every information-seeking question as Clarification.
  A question seeking new information is Answer, unless its primary purpose
  is to resolve ambiguity or repair a misunderstanding.
- Use Acknowledgment only when acknowledgment or a conventional social
  response is the primary function.
- When wording could fit multiple labels, prioritize the function most
  directly called for by the wording and the supplied Context.
- When the Previous Utterance contains multiple clauses or communicative acts,
  identify the main response-relevant question, request, or conversational act.
- Do not automatically prioritize the first or last clause. Consider the
  wording and supplied Context to identify the main communicative purpose.
- Use Context to resolve ambiguity, but do not let a minor clause override
  the main communicative purpose.
- Select only one Expected Function, even when the Previous Utterance
  contains multiple communicative acts.
- If the utterance is difficult to classify, choose the best-supported
  label from the permitted taxonomy. Do not invent a label for uncertainty.
- Do not combine labels or provide more than one Expected Function.

OUTPUT
Return only one valid JSON object with exactly this field:

{
  "expected_function": "..."
}

The value must be exactly one of:
"Answer",
"Accept/Reject",
"Perform/Provide",
"Clarification",
"Acknowledgment"

Do not output reasoning, explanations, Markdown, code fences, or any text
outside the JSON object.
"""


# ============================================================
# STAGE 2: ACTUAL RESPONSE FUNCTION AND FULFILLMENT
# ============================================================

STAGE2_SYSTEM_PROMPT = """
You are annotating actual response functions and evaluating fulfillment
in everyday English conversations.

TASK
You will receive:
- Context: an optional preceding utterance.
- Previous Utterance: the utterance to which the response is responding.
- Response: the actual response.
- Expected Function: the function determined in Stage 1.

Your tasks are to:
1. Identify the primary function of the actual Response.
2. Determine whether the Response fulfills the supplied Expected Function.
3. Assign the most appropriate Mismatch Type.
4. Provide one concise sentence explaining the labels.

DATA INDEPENDENCE
Use only the supplied dialogue text, the supplied Expected Function,
and the guidelines below.
Do not consult external sources or rely on dataset annotations, dialogue-act
labels, reference answers, metadata, or previously published classifications.
Do not infer labels from the identity or presumed source of the dialogue.

IMPORTANT
- Treat the supplied Expected Function as fixed. Do not change or reclassify it.
- Actual Function describes what the Response actually does.
- Fulfillment describes whether the Response achieves the supplied
  Expected Function.
- Mismatch Type describes the relationship between the Expected Function
  and the Response.
- These are separate decisions. Do not assign one label solely because
  another label has been assigned.
- A response may contain more than one conversational act, but select
  the single primary Actual Function.
- A response that is topically related does not automatically fulfill
  the Expected Function.
- Additional content does not invalidate an otherwise complete fulfillment.

ACTUAL FUNCTION TAXONOMY

1. Answer
Provides information, a fact, confirmation, or an explanation in response
to an information need or question.
Use Answer for both open-ended and yes/no answers.
Do not use Answer when the Response merely asks for information.

2. Accept/Reject
Accepts or refuses an invitation, offer, proposal, or request for agreement
or commitment.
Use this when acceptance or refusal is the primary function of the Response.
A response may accept or refuse and also include additional content.

3. Perform/Provide
Actually carries out a requested action or provides the requested item,
service, or resource.
Do not use this merely because the Response states willingness or agreement
to perform an action.
A response that only promises or agrees to act has not yet performed
or provided the requested action or item.

4. Clarification
Provides an explanation or correction specifically intended to resolve
ambiguity, uncertainty, misunderstanding, or an unclear reference or meaning
in a previous utterance.
Use Clarification when the Response itself clarifies what was previously
said or corrects a misunderstanding.
Do not use Clarification when the Response asks the other speaker to explain
or clarify something. Classify such questions as Information Request,
unless their primary function is to answer a previous question.

5. Acknowledgment
Primarily acknowledges, recognizes, or indicates receipt or understanding
of a previous utterance or action.
This includes conventional greetings, thanks, and apologies when
acknowledgment is the primary function.
Do not use Acknowledgment when the Response also provides substantive
information or confirmation that is the primary function.

6. Information Request
Asks the other speaker for information, an explanation, or additional details.
This includes questions seeking new information and questions requesting
clarification, such as "What do you mean?" or "Which one are you referring to?"
Use Information Request when the Response seeks information from the other
speaker, even if the question concerns something ambiguous in a previous
utterance.
Do not use Information Request when the Response itself provides the
explanation or correction.

ACTUAL FUNCTION DECISION RULES

- Classify what the Response actually does, not merely what the Previous
  Utterance requested.
- Select exactly one Actual Function label.
- Choose the primary conversational function that best represents the
  Response as a whole.
- If the Response answers a question and then offers further help, use Answer
  when the answer is the primary function.
- If the Response explicitly accepts an invitation and asks an additional
  question, use Accept/Reject when acceptance is the primary response
  to the invitation.
- If the Response asks a new information-seeking question instead of answering
  the Previous Utterance, use Information Request.
- Use Clarification when the Response itself explains or corrects something
  to resolve ambiguity or misunderstanding.
- Use Information Request when the Response asks the other speaker to provide
  information, an explanation, or additional details, including questions
  such as "What do you mean?"
- Classify the Response according to whether it provides clarification or
  requests information, not merely according to whether clarification is
  involved in the conversation.
- A response that answers a question by explaining a fact or reason should
  be classified as Answer when answering the question is its primary function,
  rather than Clarification.
- Do not label every question as Clarification.
- Do not use Perform/Provide for a response that merely describes where an
  item can be obtained or explains how to do something. Use Answer for
  information and Perform/Provide for actual performance or provision.
- When a Response contains multiple conversational acts, identify the act
  that most directly addresses the Previous Utterance as the primary function.
- Do not automatically prioritize the first or last sentence. Consider the
  communicative purpose of the Response as a whole.
- Additional questions, comments, or acknowledgments must not override
  a clearly completed primary response.
- If multiple functions are present and none clearly dominates, choose the
  function most directly relevant to the Previous Utterance.
- Do not invent labels, combine labels, or use a label outside the taxonomy.

FULFILLMENT LABELS

1. Fulfilled
The Response achieves the supplied Expected Function in the conversational
context. The fulfillment may be direct or indirect, but the expected
communicative goal must actually be achieved.
Topical relevance, politeness, or a related question alone does not count
as fulfillment.

2. Not Fulfilled
The Response does not achieve the supplied Expected Function completely.
This includes cases of no fulfillment and partial fulfillment.

3. Unjudgeable
The supplied dialogue lacks essential information needed to determine
fulfillment reliably.
Use this only when the evidence is genuinely insufficient, not simply because
the Response is unexpected, incorrect, incomplete, or unrelated.

FULFILLMENT DECISION RULES

- Evaluate fulfillment against the supplied Expected Function, not against
  general relevance, politeness, grammatical quality, or response quality.
- Determine whether the expected communicative goal was actually achieved,
  not merely whether the Response is topically related.
- Fulfillment does not require a direct or explicit answer if the expected
  communicative goal is genuinely achieved through an indirect response.
- Do not assign Fulfilled merely because the Response is relevant to the topic
  or maintains the conversation.
- A response that only asks a new or different information-seeking question
  instead of providing the expected information is normally Not Fulfilled.
- A response that only agrees or promises to perform an action does not
  fulfill a Perform/Provide expectation unless the expected function was
  to accept or commit to the action.
- For a multi-part request, use Fulfilled only when all essential requested
  parts are completed.
- If only part of a multi-part request is completed, use Not Fulfilled.
- If the Response achieves the Expected Function and adds another question,
  comment, or conversational contribution, it may still be Fulfilled.
- Do not label a Response Not Fulfilled solely because it contains additional
  content or introduces a secondary topic after fulfilling the expectation.
- Use Unjudgeable only when the supplied dialogue lacks essential evidence
  for a reliable decision.
- Disagreement between models or repeated runs is not, by itself, sufficient
  evidence for Unjudgeable. If the dialogue supports a reasonable judgment,
  choose Fulfilled or Not Fulfilled.
- If Expected Function is missing or is not one of the permitted labels,
  set Fulfillment to Unjudgeable and Mismatch Type to None.

MISMATCH TYPE TAXONOMY

1. None
The Expected Function is fulfilled, or fulfillment is Unjudgeable because
the available evidence is insufficient for a reliable decision.

2. Alternative
The Response offers a different option, substitute, or course of action
instead of fulfilling the Expected Function.
The alternative must be identifiable in the dialogue and must not itself
fully achieve the Expected Function.
Do not use Alternative merely because the Response asks a different question,
mentions another option, or contains additional information.

3. Topic Shift
The Response moves to a clearly different topic instead of addressing
the Expected Function.
Use this only when the topic change is substantial and identifiable.
Do not use Topic Shift merely because the Response fails to answer.

4. Indirect Response
The Response engages with the same request, issue, or communicative goal
but does not achieve the Expected Function.
It may address the issue partially, evade the requested action, or respond
to a related aspect without completing the expected goal.
Do not use Indirect Response solely because the Response is phrased
indirectly if it actually fulfills the Expected Function.

5. Non-Response
The Response provides no meaningful engagement with the Expected Function
and does not fit Alternative, Topic Shift, or Indirect Response.
Use this only when none of the other mismatch types adequately describes
the failure.

MISMATCH DECISION RULES

- If Fulfillment is Fulfilled, Mismatch Type MUST be None.
- If Fulfillment is Unjudgeable, Mismatch Type MUST be None.
- If Fulfillment is Not Fulfilled, select one specific mismatch type:
  Alternative, Topic Shift, Indirect Response, or Non-Response.
- Mismatch Type must never be blank.
- Use Alternative when an identifiable alternative or substitute is proposed
  instead of the expected response and the alternative does not itself
  achieve the Expected Function.
- Use Topic Shift when a substantial change of topic replaces the expected
  response.
- Use Indirect Response when the Response engages with the same request,
  issue, or communicative goal but does not achieve the Expected Function.
- Use Non-Response when the Response provides no meaningful engagement
  with the Expected Function and no more specific mismatch type applies.
- When several mismatch types appear plausible, choose the most specific
  type supported by the dialogue.
- Do not use Alternative as a general label for a different question
  or an unexpected response.
- Do not use Topic Shift as a general label for failure to fulfill.
- If the Response fulfills the Expected Function and also changes the topic
  afterward, use Fulfilled and None.

CONSISTENCY REQUIREMENTS

The following combinations are required:
- Fulfilled -> Mismatch Type = None
- Unjudgeable -> Mismatch Type = None
- Not Fulfilled -> Mismatch Type must be one of:
  Alternative, Topic Shift, Indirect Response, Non-Response

OUTPUT
Return only one valid JSON object with exactly these fields:

{
  "actual_function": "...",
  "fulfillment": "...",
  "mismatch_type": "...",
  "explanation": "..."
}

Permitted Actual Function values:
"Answer",
"Accept/Reject",
"Perform/Provide",
"Clarification",
"Acknowledgment",
"Information Request"

Permitted Fulfillment values:
"Fulfilled",
"Not Fulfilled",
"Unjudgeable"

Permitted Mismatch Type values:
"None",
"Alternative",
"Topic Shift",
"Indirect Response",
"Non-Response"

The explanation must be exactly one concise sentence.
It must not be empty.
It must justify the labels using only the supplied dialogue and Expected Function.
Do not change the supplied Expected Function.
Do not include multiple sentences, reasoning steps, or unsupported assumptions.

Do not output Markdown, code fences, headings, or any text outside
the JSON object.
"""

## Helper functions

In [ ]:
def load_model(model_id):
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required.")

    compute_dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Ensure a valid padding token is available.
    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is None:
            raise ValueError(
                f"Tokenizer for {model_id} has no pad or EOS token."
            )
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=compute_dtype
    )

    model.eval()
    return tokenizer, model


def build_stage1_prompt(row):
    context = str(row["context"]).strip()
    previous = str(row["previous_utterance"]).strip()

    return f"""
Context:
{context if context else "[No additional context provided]"}

Previous utterance:
{previous}
"""


def build_stage2_prompt(row, expected_function):
    context = str(row["context"]).strip()
    previous = str(row["previous_utterance"]).strip()
    response = str(row["response"]).strip()

    return f"""
Context:
{context if context else "[No additional context provided]"}

Previous utterance:
{previous}

Response:
{response}

Expected Function (already determined; do not change):
{expected_function}
"""


def set_run_seed(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def generate_text(
    model,
    tokenizer,
    system_prompt,
    user_prompt,
    model_name,
    seed,
    max_new_tokens=220
):
    set_run_seed(seed)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    if model_name == "Qwen3-14B":
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        inputs = tokenizer(
            formatted,
            return_tensors="pt"
        ).to(model.device)

    else:
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)

    pad_token_id = tokenizer.pad_token_id

    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id

    if pad_token_id is None:
        raise ValueError(
            "Tokenizer has no valid padding or EOS token."
        )

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=pad_token_id
        )

    # Extract only newly generated tokens, excluding the input prompt.
    input_length = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0][input_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


def parse_json_response(text):
    text = str(text).strip()

    # Remove optional Markdown code fences.
    text = re.sub(
        r"^\s*```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )
    text = re.sub(r"\s*```\s*$", "", text)

    # First try parsing the complete response.
    try:
        result = json.loads(text)
        if isinstance(result, dict):
            return result
    except json.JSONDecodeError:
        pass

    # Fallback: find the first valid JSON object.
    decoder = json.JSONDecoder()

    for match in re.finditer(r"\{", text):
        try:
            result, _ = decoder.raw_decode(text[match.start():])
            if isinstance(result, dict):
                return result
        except json.JSONDecodeError:
            continue

    raise ValueError(
        f"No valid JSON object found: {text[:500]}"
    )


def validate_stage1(result):
    if not isinstance(result, dict):
        raise ValueError("Stage 1 output must be a JSON object.")

    if set(result) != {"expected_function"}:
        raise ValueError(
            "Stage 1 must contain only expected_function."
        )

    label = result["expected_function"]

    if not isinstance(label, str) or label not in EXPECTED_LABELS:
        raise ValueError(
            f"Invalid Expected Function: {label!r}"
        )

    return True


def validate_stage2(result):
    if not isinstance(result, dict):
        raise ValueError("Stage 2 output must be a JSON object.")

    required = {
        "actual_function",
        "fulfillment",
        "mismatch_type",
        "explanation"
    }

    if set(result) != required:
        raise ValueError(
            f"Stage 2 fields must be exactly: {sorted(required)}"
        )

    actual_function = result["actual_function"]
    fulfillment = result["fulfillment"]
    mismatch_type = result["mismatch_type"]
    explanation = result["explanation"]

    if (
        not isinstance(actual_function, str)
        or actual_function not in ACTUAL_LABELS
    ):
        raise ValueError(
            f"Invalid Actual Function: {actual_function!r}"
        )

    if (
        not isinstance(fulfillment, str)
        or fulfillment not in FULFILLMENT_LABELS
    ):
        raise ValueError(
            f"Invalid Fulfillment: {fulfillment!r}"
        )

    if (
        not isinstance(mismatch_type, str)
        or mismatch_type not in MISMATCH_LABELS
    ):
        raise ValueError(
            f"Invalid Mismatch Type: {mismatch_type!r}"
        )

    if not isinstance(explanation, str) or not explanation.strip():
        raise ValueError("Explanation must be a non-empty string.")

    if "\n" in explanation.strip() or "\r" in explanation.strip():
        raise ValueError(
            "Explanation must be exactly one sentence on one line."
        )

    if fulfillment == "Fulfilled" and mismatch_type != "None":
        raise ValueError(
            "Fulfilled requires Mismatch Type = None."
        )

    if (
        fulfillment == "Not Fulfilled"
        and mismatch_type == "None"
    ):
        raise ValueError(
            "Not Fulfilled requires a specific Mismatch Type."
        )

    if fulfillment == "Unjudgeable" and mismatch_type != "None":
        raise ValueError(
            "Unjudgeable requires Mismatch Type = None."
        )

    return True


def majority_vote(values):
    valid_values = [
        str(value).strip()
        for value in values
        if pd.notna(value) and str(value).strip()
    ]

    if len(valid_values) < 2:
        return ""

    counts = Counter(valid_values)
    label, count = counts.most_common(1)[0]

    return label if count >= 2 else ""


def consensus_status(values):
    valid_values = [
        str(value).strip()
        for value in values
        if pd.notna(value) and str(value).strip()
    ]

    if len(valid_values) < 2:
        return "Incomplete"

    counts = Counter(valid_values)
    max_count = counts.most_common(1)[0][1]

    if len(valid_values) == 3 and len(set(valid_values)) == 1:
        return "Unanimous"

    if max_count >= 2:
        return "Majority"

    return "No Consensus"

## Master file

In [ ]:
def model_prefix(model_name):
    return model_name.replace("-", "_").replace(".", "_")


def stage1_columns(model_name):
    prefix = model_prefix(model_name)
    columns = {}

    for run in range(1, N_RUNS + 1):
        columns[f"{prefix}_s1_run{run}_label"] = ""
        columns[f"{prefix}_s1_run{run}_raw"] = ""
        columns[f"{prefix}_s1_run{run}_error"] = ""

    columns[f"{prefix}_s1_final"] = ""
    columns[f"{prefix}_s1_consensus_status"] = ""

    return columns


def stage2_columns(model_name):
    prefix = model_prefix(model_name)
    columns = {}

    for run in range(1, N_RUNS + 1):
        for field in [
            "actual",
            "fulfillment",
            "mismatch",
            "explanation",
            "raw",
            "error"
        ]:
            columns[f"{prefix}_s2_run{run}_{field}"] = ""

    for field in ["actual", "fulfillment", "mismatch"]:
        columns[f"{prefix}_s2_final_{field}"] = ""
        columns[f"{prefix}_s2_{field}_consensus_status"] = ""

    columns[f"{prefix}_s2_overall_status"] = ""

    return columns


# Create all expected annotation columns.
all_columns = {}

for model_name in MODELS:
    all_columns.update(stage1_columns(model_name))
    all_columns.update(stage2_columns(model_name))


# Validate the current input IDs.
if df["pair_id"].duplicated().any():
    raise ValueError(
        "Duplicate pair_id found in the current input dataset."
    )

# Normalize IDs to strings for consistent matching.
df["pair_id"] = df["pair_id"].astype(str).str.strip()

if (df["pair_id"] == "").any():
    raise ValueError(
        "Empty pair_id found in the current input dataset."
    )

if df["pair_id"].duplicated().any():
    raise ValueError(
        "Duplicate pair_id found after normalizing pair_id."
    )

current_ids = set(df["pair_id"])


# Load the existing master CSV when resuming.
if os.path.exists(OUTPUT_FILE):

    saved_df = pd.read_csv(
        OUTPUT_FILE,
        dtype=str,
        keep_default_na=False
    )

    if "pair_id" not in saved_df.columns:
        raise ValueError(
            "The existing master CSV is missing pair_id."
        )

    saved_df["pair_id"] = saved_df["pair_id"].str.strip()

    if (saved_df["pair_id"] == "").any():
        raise ValueError(
            "Empty pair_id found in the existing master CSV."
        )

    if saved_df["pair_id"].duplicated().any():
        raise ValueError(
            "Duplicate pair_id found in the existing master CSV."
        )

    saved_ids = set(saved_df["pair_id"])
    unexpected_ids = saved_ids - current_ids

    if unexpected_ids:
        raise ValueError(
            "The existing master CSV contains pair_id values "
            "that are absent from the current input dataset. "
            f"Examples: {list(unexpected_ids)[:5]}"
        )

    # Keep pair_id and previously saved annotation columns only.
    saved_annotation_cols = [
        col for col in all_columns
        if col in saved_df.columns
    ]

    saved_results = saved_df[
        ["pair_id"] + saved_annotation_cols
    ].copy()

    # Use current input data and restore saved annotations.
    master_df = df.merge(
        saved_results,
        on="pair_id",
        how="left",
        validate="one_to_one"
    )

    # Add annotation columns missing from the saved file.
    for col in all_columns:
        if col not in master_df.columns:
            master_df[col] = ""

else:
    # Start a new master file.
    master_df = df.copy()

    for col, default in all_columns.items():
        master_df[col] = default


# Fill missing annotation values in both cases.
annotation_cols = list(all_columns.keys())

master_df[annotation_cols] = (
    master_df[annotation_cols]
    .fillna("")
)


# Use pair_id as a stable lookup index.
master_df = master_df.set_index(
    "pair_id",
    drop=False,
    verify_integrity=True
)


def save_master():
    master_df.reset_index(drop=True).to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )


# Save the initialized or resumed master file.
save_master()

print("Master file:", OUTPUT_FILE)
print("Rows in master:", len(master_df))
print("Annotation columns:", len(all_columns))
print(
    "Rows with existing annotations:",
    master_df[annotation_cols].ne("").any(axis=1).sum()
)

## Annotation loop

In [ ]:
def cell_value(pair_id, column):
    return str(master_df.at[pair_id, column]).strip()


def set_value(pair_id, column, value):
    master_df.at[pair_id, column] = (
        str(value) if value is not None else ""
    )


def deterministic_seed(model_name, pair_id, stage, run_number):
    key = f"{model_name}|{pair_id}|{stage}|{run_number}"
    digest = hashlib.sha256(key.encode("utf-8")).hexdigest()
    return int(digest[:8], 16)


for model_name, model_id in MODELS.items():

    print("\n" + "=" * 80)
    print("MODEL:", model_name)
    print("=" * 80)

    tokenizer = None
    model = None

    try:
        tokenizer, model = load_model(model_id)
        prefix = model_prefix(model_name)

        for _, row in batch_df.iterrows():

            pair_id = str(row["pair_id"])

            if pair_id not in master_df.index:
                print(f"Skipping unknown pair_id: {pair_id}")
                continue

            print(f"\n{model_name} | Pair {pair_id}")

            # =================================================
            # STAGE 1: EXPECTED FUNCTION
            # =================================================

            for run_number in range(1, N_RUNS + 1):

                label_col = f"{prefix}_s1_run{run_number}_label"
                raw_col = f"{prefix}_s1_run{run_number}_raw"
                error_col = f"{prefix}_s1_run{run_number}_error"

                saved_label = cell_value(pair_id, label_col)
                saved_error = cell_value(pair_id, error_col)

                # Reuse only previously successful runs.
                if saved_label in EXPECTED_LABELS and not saved_error:
                    print(
                        f"Stage 1 run {run_number}: using saved result"
                    )
                    continue

                raw_output = ""

                try:
                    user_prompt = build_stage1_prompt(row)

                    raw_output = generate_text(
                        model=model,
                        tokenizer=tokenizer,
                        system_prompt=STAGE1_SYSTEM_PROMPT,
                        user_prompt=user_prompt,
                        model_name=model_name,
                        seed=deterministic_seed(
                            model_name, pair_id, "stage1", run_number
                        ),
                        max_new_tokens=100
                    )

                    parsed = parse_json_response(raw_output)
                    validate_stage1(parsed)

                    set_value(
                        pair_id,
                        label_col,
                        parsed["expected_function"]
                    )
                    set_value(pair_id, raw_col, raw_output)
                    set_value(pair_id, error_col, "")

                    print(
                        f"Stage 1 run {run_number}: "
                        f"{parsed['expected_function']}"
                    )

                except Exception as e:
                    set_value(pair_id, raw_col, raw_output)
                    set_value(pair_id, error_col, repr(e))
                    set_value(pair_id, label_col, "")

                    print(
                        f"Stage 1 run {run_number} ERROR: {repr(e)}"
                    )

                finally:
                    save_master()

            # Recalculate Stage 1 consensus from saved runs.
            s1_labels = [
                cell_value(
                    pair_id,
                    f"{prefix}_s1_run{run}_label"
                )
                for run in range(1, N_RUNS + 1)
            ]

            # Only valid labels count toward consensus.
            valid_s1_labels = [
                label for label in s1_labels
                if label in EXPECTED_LABELS
            ]

            s1_status = consensus_status(valid_s1_labels)
            s1_final = majority_vote(valid_s1_labels)

            previous_s1_final = cell_value(
                pair_id,
                f"{prefix}_s1_final"
            )

            # If Expected Function changes, invalidate Stage 2.
            if previous_s1_final != s1_final:
                for col in stage2_columns(model_name):
                    if col.startswith(f"{prefix}_s2_"):
                        set_value(pair_id, col, "")

            set_value(pair_id, f"{prefix}_s1_final", s1_final)
            set_value(
                pair_id,
                f"{prefix}_s1_consensus_status",
                s1_status
            )
            save_master()

            print(
                "Stage 1 labels:", s1_labels,
                "| Final:", s1_final or "No majority",
                "| Status:", s1_status
            )

            # Stage 2 requires a valid Stage 1 majority.
            if not s1_final:
                print("Stage 2 skipped: Stage 1 has no majority.")
                continue

            # =================================================
            # STAGE 2: ACTUAL FUNCTION, FULFILLMENT, MISMATCH
            # =================================================

            for run_number in range(1, N_RUNS + 1):

                base = f"{prefix}_s2_run{run_number}_"

                actual_col = base + "actual"
                fulfillment_col = base + "fulfillment"
                mismatch_col = base + "mismatch"
                explanation_col = base + "explanation"
                raw_col = base + "raw"
                error_col = base + "error"

                saved_actual = cell_value(pair_id, actual_col)
                saved_fulfillment = cell_value(
                    pair_id, fulfillment_col
                )
                saved_mismatch = cell_value(pair_id, mismatch_col)
                saved_explanation = cell_value(
                    pair_id, explanation_col
                )
                saved_error = cell_value(pair_id, error_col)

                # Validate saved labels and their logical consistency.
                saved_run_valid = (
                    saved_actual in ACTUAL_LABELS
                    and saved_fulfillment in FULFILLMENT_LABELS
                    and saved_mismatch in MISMATCH_LABELS
                    and bool(saved_explanation)
                    and not saved_error
                    and (
                        saved_fulfillment != "Fulfilled"
                        or saved_mismatch == "None"
                    )
                    and (
                        saved_fulfillment != "Not Fulfilled"
                        or saved_mismatch in {
                            "Alternative",
                            "Topic Shift",
                            "Indirect Response",
                            "Non-Response"
                        }
                    )
                    and (
                        saved_fulfillment != "Unjudgeable"
                        or saved_mismatch == "None"
                    )
                )

                if saved_run_valid:
                    print(
                        f"Stage 2 run {run_number}: using saved result"
                    )
                    continue

                raw_output = ""

                try:
                    user_prompt = build_stage2_prompt(row, s1_final)

                    raw_output = generate_text(
                        model=model,
                        tokenizer=tokenizer,
                        system_prompt=STAGE2_SYSTEM_PROMPT,
                        user_prompt=user_prompt,
                        model_name=model_name,
                        seed=deterministic_seed(
                            model_name, pair_id, "stage2", run_number
                        ),
                        max_new_tokens=220
                    )

                    parsed = parse_json_response(raw_output)
                    validate_stage2(parsed)

                    set_value(
                        pair_id, actual_col, parsed["actual_function"]
                    )
                    set_value(
                        pair_id, fulfillment_col, parsed["fulfillment"]
                    )
                    set_value(
                        pair_id, mismatch_col, parsed["mismatch_type"]
                    )
                    set_value(
                        pair_id, explanation_col, parsed["explanation"]
                    )
                    set_value(pair_id, raw_col, raw_output)
                    set_value(pair_id, error_col, "")

                    print(
                        f"Stage 2 run {run_number}: "
                        f"{parsed['actual_function']} | "
                        f"{parsed['fulfillment']} | "
                        f"{parsed['mismatch_type']}"
                    )

                except Exception as e:
                    set_value(pair_id, raw_col, raw_output)
                    set_value(pair_id, error_col, repr(e))

                    # Clear parsed fields after a failed attempt.
                    set_value(pair_id, actual_col, "")
                    set_value(pair_id, fulfillment_col, "")
                    set_value(pair_id, mismatch_col, "")
                    set_value(pair_id, explanation_col, "")

                    print(
                        f"Stage 2 run {run_number} ERROR: {repr(e)}"
                    )

                finally:
                    save_master()

            # =================================================
            # STAGE 2 CONSENSUS: VOTE PER DIMENSION
            # =================================================

            for dimension, label_set in [
                ("actual", ACTUAL_LABELS),
                ("fulfillment", FULFILLMENT_LABELS),
                ("mismatch", MISMATCH_LABELS)
            ]:

                run_labels = [
                    cell_value(
                        pair_id,
                        f"{prefix}_s2_run{run}_{dimension}"
                    )
                    for run in range(1, N_RUNS + 1)
                ]

                valid_labels = [
                    label for label in run_labels
                    if label in label_set
                ]

                final_label = majority_vote(valid_labels)
                status = consensus_status(valid_labels)

                set_value(
                    pair_id,
                    f"{prefix}_s2_final_{dimension}",
                    final_label
                )
                set_value(
                    pair_id,
                    f"{prefix}_s2_{dimension}_consensus_status",
                    status
                )

            dimension_statuses = [
                cell_value(
                    pair_id,
                    f"{prefix}_s2_{dimension}_consensus_status"
                )
                for dimension in ["actual", "fulfillment", "mismatch"]
            ]

            final_actual = cell_value(
                pair_id, f"{prefix}_s2_final_actual"
            )
            final_fulfillment = cell_value(
                pair_id, f"{prefix}_s2_final_fulfillment"
            )
            final_mismatch = cell_value(
                pair_id, f"{prefix}_s2_final_mismatch"
            )

            # All dimensions must have a majority.
            all_dimensions_have_majority = (
                all(
                    status in {"Unanimous", "Majority"}
                    for status in dimension_statuses
                )
                and final_actual in ACTUAL_LABELS
                and final_fulfillment in FULFILLMENT_LABELS
                and final_mismatch in MISMATCH_LABELS
            )

            # Final fulfillment and mismatch must be consistent.
            final_labels_consistent = (
                (
                    final_fulfillment == "Fulfilled"
                    and final_mismatch == "None"
                )
                or (
                    final_fulfillment == "Not Fulfilled"
                    and final_mismatch in {
                        "Alternative",
                        "Topic Shift",
                        "Indirect Response",
                        "Non-Response"
                    }
                )
                or (
                    final_fulfillment == "Unjudgeable"
                    and final_mismatch == "None"
                )
            )

            if not all_dimensions_have_majority:
                overall_status = "No Consensus"
            elif not final_labels_consistent:
                overall_status = "Inconsistent Consensus"
            else:
                overall_status = "Completed"

            set_value(
                pair_id,
                f"{prefix}_s2_overall_status",
                overall_status
            )
            save_master()

            print(
                "Stage 2 status:", overall_status,
                "| Actual:", final_actual,
                "| Fulfillment:", final_fulfillment,
                "| Mismatch:", final_mismatch
            )

    except Exception as e:
        print(f"Could not load/run {model_name}: {repr(e)}")

    finally:
        if model is not None:
            del model

        if tokenizer is not None:
            del tokenizer

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        save_master()

print("\nAnnotation run finished.")
print("Saved:", OUTPUT_FILE)